In [6]:
#imports

import pandas as pd
import numpy as np
from numpy import random
from numpy import linalg as LA
import seaborn as sns
import snakeviz
import line_profiler
from pathlib import Path
import os

# **Plan**

- train 3 models:
    - A benchmark logistic regression with probability output (retrieve from logit)
    - MLP with standard probabilistic MSE (Brier score)
    - MLP with weighted probabilistic MSE
- Evaluate perofrmance compared to benchmark

# **Load data**

In [9]:
path = Path.cwd().parent
data_path = path / "data"

df_X = pd.read_csv(data_path / "X_trn.csv")
df_Y = pd.read_csv(data_path / "Y_trn.csv")

In [10]:
df_X.head(3)

,X01,X02,X03,X04,X05,X06,X07,X08,X09,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
0,1514,30177,-10,1,26,-5,-4,0,1890,33490,...,3835,-2,2,-3,28439,51533,3097,1896,22,35014
1,0,0,-16,2,51,-29,-16,0,0,0,...,0,-12,1,-23,0,296630,0,0,-2,0
2,0,2945,-16,1,39,-23,-13,1216,2997,2675,...,2470,-10,1,-19,1271,306063,0,0,2,-4


In [ ]:
print(df_X.shape)
print(df_Y.shape)

(25000, 23)
(25000, 1)


# **Data Exploration**

In [ ]:
df_Y["Y"].value_counts().sort_index().to_frame("count").assign(
    proportion=lambda x: round(x["count"] / len(df_Y,), 2)
)

## The dataset is imbalanced, with a majority of the samples belonging to class 0.
## With regular ML models, rebalancing is favored, but with neural networks it may not be neccessary. Will check later.

,count,proportion
Y,,
0,19470,0.78
1,5530,0.22


In [22]:
df_X.describe()

,X01,X02,X03,X04,X05,X06,X07,X08,X09,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
count,25000.00000,2.500000e+04,25000.000000,25000.000000,25000.00000,25000.000000,25000.000000,25000.000000,2.500000e+04,25000.000000,...,25000.000000,25000.000000,25000.000000,25000.000000,2.500000e+04,2.500000e+04,25000.000000,25000.000000,25000.000000,25000.000000
mean,5291.73684,4.718632e+04,-19.047160,1.600640,35.47872,-18.309920,-10.490080,5652.883040,5.870609e+03,40517.481720,...,4867.042880,-8.526800,1.551400,-15.524800,4.934852e+04,1.674288e+05,5247.070200,4832.123000,4.841600,39069.430360
std,18016.36868,6.991226e+04,3.368565,0.489777,9.25810,7.026897,3.600414,15850.257553,2.246669e+04,61184.642714,...,15974.451874,2.276552,0.522235,4.789929,7.170667e+04,1.307733e+05,17226.376558,15408.337477,4.621247,59809.027104
min,0.00000,-1.644070e+05,-25.000000,1.000000,21.00000,-29.000000,-16.000000,0.000000,0.000000e+00,-80654.000000,...,0.000000,-12.000000,0.000000,-23.000000,-7.187600e+04,8.819000e+03,0.000000,0.000000,-2.000000,-309891.000000
25%,120.00000,2.737000e+03,-22.000000,1.000000,28.00000,-23.000000,-13.000000,965.000000,8.600000e+02,1801.500000,...,291.000000,-10.000000,1.000000,-19.000000,2.999750e+03,5.386775e+04,412.000000,274.000000,2.000000,1276.500000
50%,1502.00000,2.026500e+04,-19.000000,2.000000,34.00000,-17.000000,-10.000000,2148.000000,2.083000e+03,18020.000000,...,1510.000000,-8.000000,2.000000,-15.000000,2.137300e+04,1.393640e+05,1826.000000,1546.000000,6.000000,17147.500000
75%,4093.00000,6.038675e+04,-19.000000,2.000000,41.00000,-17.000000,-10.000000,5111.250000,5.040000e+03,50993.250000,...,4086.000000,-8.000000,2.000000,-15.000000,6.390675e+04,2.389145e+05,4516.250000,4121.250000,6.000000,49882.750000
max,535570.00000,1.749017e+06,5.000000,2.000000,78.00000,31.000000,14.000000,487371.000000,1.678842e+06,896445.000000,...,577660.000000,8.000000,3.000000,17.000000,1.087738e+06,1.017656e+06,895321.000000,442282.000000,38.000000,859923.000000
